In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))
from data_tools import process_data as processed
from data_tools.combine_data import CreateTrainingandTestData
from data_tools import process_dates
from data_tools import handle_datetime
from data_tools import clean_data as clean

combine = CreateTrainingandTestData()
# Change as needed
sensor_id = "Tranquility"
pmdata = "m_PM25_CF1"
year = "2023"

data_name = {
    "m_PM25_CF1":{
        "temp": "tempC_pms",
        "rh": "rh_pms",
        "pm1": "m_PM1_CF1",
        "pm2.5": "m_PM25_CF1",
        "pm10": "m_PM10_CF1",
        "name": "Plantower"
    },
    "m_PM25_b":{
        "temp": "tempC_sen5x",
        "rh": "rh_sen5x",
        "pm1": "m_PM1_b",
        "pm2.5": "m_PM25_b",
        "pm10": "m_PM10_b",
        "name": "Sensirion"
    }
}
sensor = data_name[pmdata]
training_dates = process_dates.training(sensor_id)
testing_dates= process_dates.testing(sensor_id)

# Define files that hold necessary raw data
reference_file_1 = rf"../reference_files/PM25HR_PICKDATA_2023-12-31-Fresno.csv"
if sensor_id == "Tranquility":
    reference_file_2 = rf"../reference_files/PM25HR_PICKDATA_2023-12-31-Tranquility.csv"
else:
    reference_file_2 = reference_file_1
sensor_file = rf"../reference_files/2023rawdata/{sensor_id}.csv"
calibration_reference = handle_datetime.utc_to_CA(processed.ref_data(reference_file_1))
deployment_reference = handle_datetime.utc_to_CA(processed.ref_data(reference_file_2))
voz_data = handle_datetime.utc_to_CA(processed.raw_voz_data(sensor_file))

when = {
    "precal_start": training_dates[0],
    "precal_end": training_dates[1],
    "postcal_start": training_dates[2],
    "postcal_end": training_dates[3],
    "trial1_start": testing_dates[0],
    "trial1_end": testing_dates[1],
    "trial2_start": testing_dates[2],
    "trial2_end": testing_dates[3]
}
combine.set_calibration_parameters(pmdata,sensor_id,voz_data)
training_data,all_data = combine.get_combined_data(training_dates,testing_dates,calibration_reference,deployment_reference)
training_data, all_data = map(lambda df: clean.eliminate_waste_data(df, sensor), [training_data, all_data])